In [1]:
# %pip install langgraph langchain-openai pydantic

In [2]:
from typing import Any, Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict


class RiskReport(BaseModel):
    order_id: str = Field(description="The order ID extracted from ticket")
    user_id: str = Field(description="The user ID attached to the order")
    risk_score: int = Field(description="Fraud risk score 0-100 from audit_fraud_risk")
    risk_band: Literal["low", "medium", "high"] = Field(description="Risk category level")
    triggered_rules: list[dict[str, Any]] = Field(
        default_factory=list, description="Rules triggered during audit"
    )
    blocks_automatic_refund: bool = Field(description="True if risk_score > 70 or high risk band")
    error: str | None = Field(default=None, description="Error code if investigation failed")
    evidence: dict[str, Any] = Field(default_factory=dict, description="Raw signals from audit")


class Decision(BaseModel):
    order_id: str
    user_id: str
    verdict: Literal["ELIGIBLE", "INELIGIBLE", "ESCALATION_REQUIRED"] = Field(
        description="Return policy check result"
    )
    refund_status: Literal["APPROVED", "REJECTED", "ESCALATION_REQUIRED"] = Field(
        description="Final operational refund status"
    )
    requested_amount: float = Field(default=0.0, description="Refund amount requested")
    approved_amount: float = Field(default=0.0, description="Refund amount approved")
    risk_band: str = Field(description="Carried over from RiskReport")
    rationale: str = Field(description="Internal business logic explaining the decision")


class FinalResolution(BaseModel):
    customer_response: str = Field(
        description="Polite message to customer. DO NOT disclose fraud flags."
    )
    slack_alert_sent: bool = Field(
        description="True if an alert was posted to outbox/alerts.jsonl"
    )
    escalation_channel: str | None = Field(default=None, description="Target channel if escalated")


class AgentState(TypedDict):
    ticket_text: str
    risk_report: RiskReport | None
    decision: Decision | None
    final_resolution: FinalResolution | None

In [3]:
import multi_agent_tools as mat

researcher_tools = mat.RESEARCHER_TOOLS
decision_tools = mat.DECISION_TOOLS
comms_tools = mat.COMMS_TOOLS

assert "process_refund" not in {t["name"] for t in comms_tools}, (
    "Violation: Comms agent cannot have process_refund"
)
assert "send_slack_alert" not in {t["name"] for t in researcher_tools}, (
    "Violation: Researcher cannot send alerts"
)
assert "process_refund" not in {t["name"] for t in researcher_tools}, (
    "Violation: Researcher cannot execute refunds"
)
print("✓ Tool Isolation Guardrails Verified Successfully.")

✓ Tool Isolation Guardrails Verified Successfully.


In [ ]:
import time

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:
def researcher_node(state: AgentState) -> dict:
    ticket = state["ticket_text"]

    class ExtractedTicket(BaseModel):
        order_id: str
        claimed_user_id: str | None = None

    extractor = llm.with_structured_output(ExtractedTicket)
    extracted = extractor.invoke(
        f"Extract the order_id (e.g. ORD-1001) and any mentioned user_id (e.g. USR-101) from this ticket: {ticket}"
    )
    order_id = extracted.order_id
    claimed_user_id = extracted.claimed_user_id

    order_details = mat.get_order_details(order_id)

    if "error" in order_details:
        risk_report = RiskReport(
            order_id=order_id,
            user_id=claimed_user_id or "UNKNOWN",
            risk_score=0,
            risk_band="low",
            triggered_rules=[{"rule_id": "ORDER_NOT_FOUND"}],
            blocks_automatic_refund=False,
            error="ORDER_NOT_FOUND",
        )
        return {"risk_report": risk_report}

    user_id = order_details["user_id"]
    user_profile = mat.get_user_profile(user_id)
    fraud_audit = mat.audit_fraud_risk(order_id, claimed_user_id or user_id)

    if "error" in fraud_audit:
        risk_report = RiskReport(
            order_id=order_id,
            user_id=claimed_user_id or user_id,
            risk_score=100,
            risk_band="high",
            triggered_rules=[],
            blocks_automatic_refund=True,
            error=fraud_audit["error"],
            evidence={
                "order_total_usd": order_details.get("order_total_usd", 0.0),
                "user_profile": user_profile,
            },
        )
        return {"risk_report": risk_report}

    risk_score = fraud_audit.get("risk_score", 0)
    risk_band = fraud_audit.get("risk_band", "low")
    blocks = fraud_audit.get("blocks_automatic_refund", risk_score > 70 or risk_band == "high")

    evidence = fraud_audit.get("evidence", {})
    evidence["user_profile"] = user_profile

    risk_report = RiskReport(
        order_id=order_id,
        user_id=user_id,
        risk_score=risk_score,
        risk_band=risk_band,
        triggered_rules=fraud_audit.get("triggered_rules", []),
        blocks_automatic_refund=blocks,
        evidence=evidence,
    )

    time.sleep(1)
    return {"risk_report": risk_report}

In [6]:
def decision_node(state: AgentState) -> dict:
    risk_report = state["risk_report"]
    ticket = state["ticket_text"]

    if risk_report.error:
        is_mismatch = risk_report.error == "USER_ORDER_MISMATCH"
        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict="ESCALATION_REQUIRED" if is_mismatch else "INELIGIBLE",
            refund_status="ESCALATION_REQUIRED" if is_mismatch else "REJECTED",
            requested_amount=0.0,
            approved_amount=0.0,
            risk_band="high" if is_mismatch else risk_report.risk_band,
            rationale=f"Error during research: {risk_report.error}",
        )
        return {"decision": decision}

    class ReturnReason(BaseModel):
        reason: Literal[
            "damaged_on_arrival",
            "wrong_item",
            "item_missing",
            "late_delivery",
            "changed_mind",
        ]

    reason_extracted = (
        llm.with_structured_output(ReturnReason)
        .invoke(f"Identify the return reason from ticket: {ticket}")
        .reason
    )

    policy_res = mat.check_return_policy(risk_report.order_id, reason_extracted)
    verdict = policy_res.get("verdict", "INELIGIBLE")
    requested_amount = risk_report.evidence.get("order_total_usd", 0.0)

    if risk_report.blocks_automatic_refund:
        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="ESCALATION_REQUIRED",
            requested_amount=requested_amount,
            approved_amount=0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Refund blocked due to fraud risk audit (Score: {risk_report.risk_score}). Policy was: {verdict}.",
        )
    elif verdict == "ELIGIBLE":
        refund_res = mat.process_refund(risk_report.order_id, requested_amount, reason_extracted)
        status = refund_res.get("status", "REJECTED")

        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="APPROVED" if status == "APPROVED" else status,
            requested_amount=requested_amount,
            approved_amount=requested_amount if status == "APPROVED" else 0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Refund processed with status: {status}.",
        )
    else:
        decision = Decision(
            order_id=risk_report.order_id,
            user_id=risk_report.user_id,
            verdict=verdict,
            refund_status="REJECTED",
            requested_amount=requested_amount,
            approved_amount=0.0,
            risk_band=risk_report.risk_band,
            rationale=f"Ineligible based on policy: {policy_res.get('details')}",
        )

    return {"decision": decision}


In [7]:
def comms_node(state: AgentState) -> dict:
    risk_report = state["risk_report"]
    decision = state["decision"]

    if risk_report.error == "ORDER_NOT_FOUND":
        customer_msg = f"I'm sorry, but I couldn't find an order with the ID {decision.order_id}. Could you please double-check the order number and let me know?"
        return {
            "final_resolution": FinalResolution(
                customer_response=customer_msg, slack_alert_sent=False, escalation_channel=None
            )
        }
    prior_fraud_flags = risk_report.evidence.get("prior_fraud_flags", 0)
    order_status = risk_report.evidence.get("order_status", "unknown")

    route_info = mat.get_escalation_route(
        risk_band=decision.risk_band,
        requested_amount=decision.requested_amount,
        prior_fraud_flags=prior_fraud_flags,
        order_status=order_status,
        verdict=decision.verdict,
    )

    slack_sent = False
    escalation_channel = None

    if route_info.get("escalation_required"):
        channel_id = route_info["channel_id"]
        severity = route_info["severity"]
        escalation_channel = route_info["channel"]

        payload = {
            "order_id": decision.order_id,
            "user_id": decision.user_id,
            "risk_score": risk_report.risk_score,
            "risk_band": risk_report.risk_band,
            "triggered_rules": risk_report.triggered_rules,
            "requested_amount": decision.requested_amount,
            "evidence": risk_report.evidence,
            "rationale": decision.rationale,
        }
        alert_res = mat.send_slack_alert(
            channel_id=channel_id,
            severity=severity,
            payload=payload,
        )
        slack_sent = alert_res.get("delivered", False)

    prompt = f"""
    Formulate a polite, professional response to the customer for order {decision.order_id}.
    Decision Status: {decision.refund_status}.
    Approved Amount: ${decision.approved_amount}.

    STRICT GUARDRAIL:
    - DO NOT mention terms like "fraud", "risk score", "suspicious activity", or "audit".
    - If status is ESCALATION_REQUIRED or if an escalation alert was sent, explain that the request requires manual verification by our specialized team and we will follow up shortly.
    """

    customer_msg = llm.invoke(prompt).content.strip()

    final_res = FinalResolution(
        customer_response=customer_msg,
        slack_alert_sent=slack_sent,
        escalation_channel=escalation_channel,
    )

    return {"final_resolution": final_res}

In [8]:
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("decision", decision_node)
workflow.add_node("comms", comms_node)
workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "decision")
workflow.add_edge("decision", "comms")
workflow.add_edge("comms", END)
app = workflow.compile()
print("--- 1. Testing Fraud Scenario (ORD-1005) ---")
res_1005 = app.invoke(
    {"ticket_text": "Hi, my order ORD-1005 arrived broken. I demand a full refund now."}
)
print("\n[Risk Report]:", res_1005["risk_report"].model_dump())
print("\n[Decision]:", res_1005["decision"].model_dump())
print("\n[Final Resolution]:", res_1005["final_resolution"].model_dump())
print("Slack Alert Sent:", res_1005["final_resolution"].slack_alert_sent)
print("Escalation Channel:", res_1005["final_resolution"].escalation_channel)
print("\n--- 2. Testing Clean Eligible Scenario (ORD-1001) ---")
res_1001 = app.invoke({"ticket_text": "Hello, ORD-1001 item is damaged, please refund."})
print("\n[Risk Report]:", res_1001["risk_report"].model_dump())
print("\n[Decision]:", res_1001["decision"].model_dump())
print("\n[Final Resolution]:", res_1001["final_resolution"].model_dump())
print("Approved Amount:", res_1001["decision"].approved_amount)
print("Slack Alert Sent (Should be False):", res_1001["final_resolution"].slack_alert_sent)
print("\n--- 3. Testing Non-existent Order (ORD-0000) ---")
res_0000 = app.invoke({"ticket_text": "Refund for ORD-0000 please."})
print("\n[Risk Report]:", res_0000["risk_report"].model_dump())
print("\n[Decision]:", res_0000["decision"].model_dump())
print("\n[Final Resolution]:", res_0000["final_resolution"].model_dump())
print("[Customer Response]:", res_0000["final_resolution"].customer_response)

--- 1. Testing Fraud Scenario (ORD-1005) ---

[Risk Report]: {'order_id': 'ORD-1005', 'user_id': 'USR-105', 'risk_score': 90, 'risk_band': 'high', 'triggered_rules': [{'rule_id': 'FR-01', 'name': 'repeat_refund_claims', 'weight': 25, 'why': '3 claims in the last 60 days'}, {'rule_id': 'FR-02', 'name': 'last_minute_address_change', 'weight': 20, 'why': 'address changed 2 day(s) before delivery'}, {'rule_id': 'FR-04', 'name': 'prior_fraud_flag', 'weight': 15, 'why': '1 prior fraud flag(s) on record'}, {'rule_id': 'FR-05', 'name': 'high_initial_fraud_score', 'weight': 15, 'why': 'inherited fraud score is 61'}, {'rule_id': 'FR-08', 'name': 'refund_amount_velocity', 'weight': 15, 'why': '451.00 USD refunded in the last 60 days'}], 'blocks_automatic_refund': True, 'error': None, 'evidence': {'reference_date': '2026-08-05', 'order_total_usd': 480.0, 'order_status': 'delivered', 'claims_in_last_60_days': 3, 'refunded_usd_in_last_60_days': 451.0, 'account_age_days': 119, 'prior_fraud_flags': 1,